# Section 2: Building Grounded RAG Pipeline (55 Mins)

**Enterprise Use Case:** CloudCorp Smart Assistant (AskHR Agent)

**Notebook Goal:** Implement semantic section chunking (`\n---`), compute asymmetric embeddings using `text-embedding-004` (`RETRIEVAL_DOCUMENT` vs `RETRIEVAL_QUERY`), execute in-memory cosine vector search, and generate strictly grounded answers with `gemini-3.5-flash-lite`.

In [1]:
# Step 2.0: Install Required Google Cloud Client Libraries
!pip install -q google-cloud-aiplatform>=1.60.0 google-cloud-storage>=2.14.0 numpy


zsh:1: 1.60.0 not found


In [2]:
import numpy as np
# Step: Authenticate & Configure GCP Project (Colab OAuth or Local Service Account Key from .env)
import os

# 1. Load environment constants dynamically from .env file if available
def load_env_file():
    for path in [".env", "../.env", "cloudcorp-llms-on-gcp-lab/.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        val = v.strip().strip('"').strip("'")
                        if val:
                            os.environ.setdefault(k.strip(), val)
            print(f"Loaded configuration from '{path}'")
            return

load_env_file()

# 2. Project Configuration & Quota Project Settings
PROJECT_ID = os.environ.get("PROJECT_ID", "").strip() or os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or "your-gcp-project-id"  # @param {type:"string"}
REGION = os.environ.get("REGION", "us-central1").strip()               # @param {type:"string"}
BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-cloudcorp-policies").strip()

# Guardrail: Prevent running with unconfigured placeholder Project ID
if PROJECT_ID in ["your-gcp-project-id", "YOUR_PROJECT_ID", ""]:
    raise ValueError(
        "⚠️ ACTION REQUIRED: Please enter your Google Cloud Project ID in the PROJECT_ID field above! "
        "Ensure Vertex AI and Cloud Storage APIs are enabled."
    )

# Prevent 403 ADC Quota Project error (e.g. consumer projects/618104708054 in Cloud Shell)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID

# 3. Authentication Option A: Google Colab OAuth
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated via Google Colab OAuth.")
except ImportError:
    print("ℹ️ Running outside Google Colab (Cloud Shell / Local Machine / IDE).")

# 4. Authentication Option B: Service Account JSON Key via .env / Local File
sa_key_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "")
if sa_key_path:
    candidate_key_paths = [
        sa_key_path,
        os.path.join("..", sa_key_path),
        os.path.join("cloudcorp-llms-on-gcp-lab", sa_key_path),
        os.path.expanduser(f"~/Documents/{sa_key_path}"),
        os.path.expanduser(f"~/Downloads/{sa_key_path}")
    ]
    resolved_key = next((p for p in candidate_key_paths if os.path.isfile(p)), None)
    if resolved_key:
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(resolved_key)
        print(f"🔑 Loaded Service Account Key: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")
    else:
        print(f"⚠️ Key file '{sa_key_path}' specified in .env not found on disk.")

# 5. Verify Active Credentials & Bind Quota Project
import google.auth
credentials = None
try:
    credentials, auth_project = google.auth.default(quota_project_id=PROJECT_ID)
    if hasattr(credentials, "with_quota_project"):
        credentials = credentials.with_quota_project(PROJECT_ID)
    account_id = getattr(credentials, "service_account_email", None) or getattr(credentials, "signer_email", "Google User Account (OAuth/ADC)")
    print(f"✅ Active Identity: {account_id}")
    print(f"✅ Quota Project bound: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️ Warning: No active credentials detected ({e})")
    print("   • In Colab: Run auth.authenticate_user()")
    print("   • In Cloud Shell: Run 'gcloud auth application-default set-quota-project '")
    print("   • Locally: Set GOOGLE_APPLICATION_CREDENTIALS in .env or run 'gcloud auth application-default login'")

# 6. Initialize Vertex AI & Cloud Storage with explicit credentials
import vertexai
vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
print(f"✅ Vertex AI initialized successfully for project '{PROJECT_ID}' in '{REGION}'.")

from google.cloud import storage
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput
from vertexai.generative_models import GenerativeModel

storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = storage_client.bucket(BUCKET_NAME)

embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")
vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
llm = GenerativeModel("gemini-3.5-flash-lite")
print(f"✅ Vertex AI models loaded successfully for project '{PROJECT_ID}'.")


Loaded configuration from '../.env'
ℹ️ Running outside Google Colab (Local Machine / IDE).
🔑 Loaded Service Account Key: /Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/notebooks/gcp-key.json
✅ Active Identity: cloudcorp-lab-sa@upgradlabs-1751346914784.iam.gserviceaccount.com
✅ Vertex AI initialized successfully for project 'upgradlabs-1751346914784' in 'us-central1'.


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


✅ Vertex AI models loaded successfully for project 'upgradlabs-1751346914784'.


/Users/ujalajha/Documents/LLM on GCP/cloudcorp-llms-on-gcp-lab/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [9]:
# Step 2.2: Semantic Chunking along Section Delimiters (\n---)
def chunk_document(source_name: str, text: str):
    sections = text.split("\n---")
    results = []
    for idx, sec in enumerate(sections):
        cleaned = sec.strip()
        if len(cleaned) > 20:
            results.append({
                "source": source_name,
                "chunk_id": f"{source_name}#sec{idx+1}",
                "text": cleaned
            })
    return results

# Mandatorily load policy documents directly from Google Cloud Storage bucket
print(f"Reading policy documents from GCS bucket: gs://{BUCKET_NAME}...")
blobs = [b for b in bucket.list_blobs() if b.name.endswith(".txt")]

if not blobs:
    raise RuntimeError(
        f"❌ No policy files found in GCS bucket gs://{BUCKET_NAME}!\n"
        "Please run Section 1 (01_Section1_Setup_and_Corpus.ipynb) first to ingest the policy corpus into GCS."
    )

all_chunks = []
for blob in blobs:
    content = blob.download_as_text()
    chunks = chunk_document(blob.name, content)
    all_chunks.extend(chunks)
    print(f" • Loaded '{blob.name}' ({blob.size} bytes) from GCS -> {len(chunks)} semantic sections")

print(f"\n✅ Successfully loaded {len(all_chunks)} semantic chunks from GCS bucket.")
for c in all_chunks[:3]:
    print(f"   -> [{c['chunk_id']}]: {c['text'][:80]}...")


Reading policy documents from GCS bucket: gs://upgradlabs-1751346914784-cloudcorp-policies...
 • Loaded 'hr_pto_policy.txt' (6139 bytes) from GCS -> 9 semantic sections
 • Loaded 'it_laptop_policy.txt' (1627 bytes) from GCS -> 4 semantic sections
 • Loaded 'remote_work_policy.txt' (1374 bytes) from GCS -> 4 semantic sections

✅ Successfully loaded 17 semantic chunks from GCS bucket.
   -> [hr_pto_policy.txt#sec1]: # CloudCorp Global Human Resources Policy: Time Off, Leave of Absence, and Workp...
   -> [hr_pto_policy.txt#sec2]: ### Section 1: Paid Time Off (PTO) Accrual & Eligibility
1.1 Full-Time Regular E...
   -> [hr_pto_policy.txt#sec3]: ### Section 2: PTO Carry-Over, Expiration, and Separation Rules
2.1 Carry-Over C...


In [13]:
# Step 2.3: Asymmetric Batch Vector Embedding Generation (RETRIEVAL_DOCUMENT)
# Using Vertex AI TextEmbeddingInput with task_type="RETRIEVAL_DOCUMENT" for document indexing
import numpy as np
from vertexai.language_models import TextEmbeddingInput

doc_inputs = [
    TextEmbeddingInput(text=c["text"], task_type="RETRIEVAL_DOCUMENT")
    for c in all_chunks
]
embeddings = embedding_model.get_embeddings(doc_inputs)

for i, emb in enumerate(embeddings):
    all_chunks[i]["vector"] = np.array(emb.values)

print(f"Generated embeddings for {len(all_chunks)} chunks using task_type='RETRIEVAL_DOCUMENT'.")
print(f"Vector dimensionality: {len(all_chunks[0]['vector'])} dimensions (text-embedding-004).")


Generated embeddings for 17 chunks using task_type='RETRIEVAL_DOCUMENT'.
Vector dimensionality: 768 dimensions (text-embedding-004).


In [14]:
# Step 2.4: Cosine Similarity Vector Retrieval with task_type='RETRIEVAL_QUERY'
import numpy as np
from vertexai.language_models import TextEmbeddingInput

def retrieve_context(query: str, top_k: int = 2):
    # Generate query vector with asymmetric task_type="RETRIEVAL_QUERY"
    query_input = TextEmbeddingInput(text=query, task_type="RETRIEVAL_QUERY")
    q_vec = np.array(embedding_model.get_embeddings([query_input])[0].values)
    
    scored = []
    for chunk in all_chunks:
        dot = np.dot(q_vec, chunk["vector"])
        norm = np.linalg.norm(q_vec) * np.linalg.norm(chunk["vector"])
        sim = float(dot / norm) if norm > 0 else 0.0
        scored.append((sim, chunk))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

# Test retrieval
test_query = "How many days of PTO can I carry over, and when do they expire?"
top_hits = retrieve_context(test_query, top_k=2)
print(f"Top match for '{test_query}':")
for score, hit in top_hits:
    print(f"Score {score:.4f} | Chunk: {hit['chunk_id']}")


Top match for 'How many days of PTO can I carry over, and when do they expire?':
Score 0.7384 | Chunk: hr_pto_policy.txt#sec3
Score 0.6743 | Chunk: hr_pto_policy.txt#sec2


In [30]:
# Step 2.5: Grounded Response Generation with Gemini 3.5 Flash Lite
import vertexai
from vertexai.generative_models import GenerativeModel

# Initialize Gemini on the global Vertex AI endpoint with credentials
vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
llm = GenerativeModel("gemini-3.5-flash-lite")

def ask_cloudcorp_assistant(query: str, top_k: int = 2):
    # 1. Retrieve context using regional embeddings (us-central1)
    vertexai.init(project=PROJECT_ID, location=REGION, credentials=credentials)
    top_hits = retrieve_context(query, top_k=top_k)
    context_str = "

".join([f"[{h[1]['chunk_id']}]:
{h[1]['text']}" for h in top_hits])
    
    # 2. Generate grounded answer using Gemini 3.5 Flash Lite (global)
    vertexai.init(project=PROJECT_ID, location="global", credentials=credentials)
    prompt = f"""You are CloudCorp's AskHR Assistant. Answer the employee's question strictly using ONLY the policy context below.
Always cite the chunk_id as your source citation. If the answer cannot be found in the context, state that you do not know.

Context:
{context_str}

Question: {query}
Answer:"""
    response = llm.generate_content(prompt)
    return response.text.strip()

# Evaluate across the 3 test queries
queries = [
    "How many days of PTO can I carry over, and when do they expire?",
    "Do I need to submit a doctor's certificate if I take sick leave for 4 days?",
    "Can I take 2 weeks off during the end of December?"
]

for q in queries:
    print(f"
❓ QUESTION: {q}")
    ans = ask_cloudcorp_assistant(q)
    print(f"🤖 GROUNDED ANSWER:
{ans}
")



❓ QUESTION: How many days of PTO can I carry over, and when do they expire?
🤖 GROUNDED ANSWER:
Employees are permitted to carry over a maximum of 5 unused PTO days (40 hours) from the prior calendar year into the new calendar year. All carried-over PTO days must be scheduled and taken on or before March 31 (end of Q1) of the subsequent calendar year, and any remaining unused hours on April 1 are permanently forfeited [hr_pto_policy.txt#sec3].


❓ QUESTION: Do I need to submit a doctor's certificate if I take sick leave for 4 days?
🤖 GROUNDED ANSWER:
Yes, if you are absent due to illness for more than three (3) consecutive business days, an official medical certificate signed by a licensed healthcare professional must be uploaded to CloudCorp Benefits within 48 hours of your return to work.

Source Citation: [hr_pto_policy.txt#sec5]


❓ QUESTION: Can I take 2 weeks off during the end of December?
🤖 GROUNDED ANSWER:
Based on CloudCorp's HR policy, PTO requests spanning more than 2 conse